# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedprakash016/flyrank-ml-2026/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane: **CTR / Engagement Opportunity Scoring**

I chose this lane because the starter dataset reveals a strong signal: pages at positions 1–3 receive an average CTR of 2.71%, while pages ranked 51–100 receive only 0.15% — an 18x difference observed directly in the data. This means there is a real, measurable gap between pages that rank well and those that do not convert their visibility into clicks. The lane asks me to build a scored, ranked list of pages where improving content could recover lost engagement — a question with a clear decision, a clear action, and a clear cost if I get it wrong.

In [5]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Vedprakash016/flyrank-ml-2026"
REPO_DIR = "flyrank-ml-2026"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset: {df.shape[0]:,} pages, {df.shape[1]} columns")
print(df[["impressions_90d","ctr","avg_position","trend_direction"]].describe())

Dataset: 30,000 pages, 44 columns
       impressions_90d           ctr  avg_position
count     30000.000000  30000.000000   30000.00000
mean       5200.366300      0.510733      16.34238
std       16838.019547      3.279162      15.21679
min           1.000000      0.000000       0.00000
25%          81.000000      0.000000       6.20000
50%         731.000000      0.070000      10.80000
75%        3615.250000      0.290000      22.30000
max      517715.000000    100.000000     245.00000


**Research question:** Which pages have the highest untapped CTR potential — pages that rank well enough to be seen but are not being clicked?

**Unit of analysis:** One page (one content item in the dataset).

**Output:** A ranked CTR opportunity score per page, highest priority first.

**Decision it improves:** Which pages should a content team prioritize for optimization this sprint?

**Action someone takes:** A content editor rewrites titles or meta descriptions for top-ranked pages to improve click-through rate.

**Cost of a wrong recommendation:**
- False positive: wasted editor time on a page that was already performing well.
- False negative: the page keeps losing clicks it could have had — lost traffic and conversions.

**Why data / ML helps:** A human cannot manually score 30,000+ pages. A hand rule like "flag CTR < 1%" ignores position context — 0.5% CTR is excellent at position 20 but poor at position 3. A model can find joint thresholds a simple rule cannot.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
df_active = df[df["impressions_90d"] > 0].copy()

# Number 1: CTR by position tier
df_active["position_tier"] = pd.cut(
    df_active["avg_position"],
    bins=[0, 3, 10, 20, 50, 100],
    labels=["1-3 (Top)", "4-10 (Page 1)", "11-20 (Page 2)", "21-50 (Low)", "51-100 (Very Low)"]
)
ctr_by_tier = df_active.groupby("position_tier", observed=True)["ctr"].mean().round(4)
print("=== Number 1: Average CTR by Position Tier ===")
print(ctr_by_tier.to_string())

# Number 2: High-visibility but low-CTR pages
high_visibility = df_active[df_active["impressions_90d"] >= 500]
low_ctr = high_visibility[high_visibility["ctr"] < high_visibility["ctr"].quantile(0.25)]
print(f"\n=== Number 2: High-visibility, low-CTR pages ===")
print(f"Pages with 500+ impressions AND bottom-quartile CTR: {len(low_ctr):,}")

# Number 3: Correlation
corr = df_active["avg_position"].corr(df_active["ctr"])
print(f"\n=== Number 3: Correlation (avg_position vs ctr) ===")
print(f"Correlation: {corr:.3f}  — directional but non-linear, so a model beats a hand rule.")

=== Number 1: Average CTR by Position Tier ===
position_tier
1-3 (Top)            2.7143
4-10 (Page 1)        0.6510
11-20 (Page 2)       0.3234
21-50 (Low)          0.2223
51-100 (Very Low)    0.1525

=== Number 2: High-visibility, low-CTR pages ===
Pages with 500+ impressions AND bottom-quartile CTR: 4,136

=== Number 3: Correlation (avg_position vs ctr) ===
Correlation: -0.073  — directional but non-linear, so a model beats a hand rule.


**What I can observe:**
- Pages at positions 1–3 show directionally higher CTR than pages at 51–100 in this dataset.
- There is a measurable group of high-impression, low-CTR pages — an observable gap.

**What I cannot claim:**
- I cannot prove that optimizing these pages will cause CTR to improve — correlation is not causation.
- I cannot claim this reflects Google's internal ranking logic.

**What the model will do:**
- Produce a ranked priority list for content editors — not make editorial decisions itself.
- Use only features known before the decision point — no leakage.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.